# CHiRPE Tutorial: Build Fused ONNX Directly

This tutorial explains, step by step, how to build a **single fused ONNX model** for CHiRPE:

- input: raw text string
- output: classifier logits

We do everything directly in notebook code (no `scripts/` calls).

## ONNX in plain language

If you are new to ONNX, think about it like this:

- A PyTorch model is Python code + weights.
- An ONNX model is a portable computation graph + weights.
- ONNX Runtime can run this graph without needing your original training code.

In CHiRPE, we need two parts for inference:

1. **Tokenizer**: turns raw text into token IDs and masks
2. **Classifier**: turns token tensors into logits

A fused model combines both parts so serving is simpler.

## What we will build

Pipeline:

`text -> tokenizer ONNX -> token tensors -> adapter -> classifier ONNX -> logits`

At the end, we will save:

- `classifier.onnx`
- `tokenizer.onnx`
- `fused.onnx`

In [1]:
from pathlib import Path
import inspect
import json

import numpy as np
import onnx
import onnx.compose
import onnxruntime as ort
import torch
from onnx import TensorProto, helper
from onnxruntime_extensions import gen_processing_models, get_library_path
from transformers import AutoModelForSequenceClassification, AutoTokenizer

2026-04-23 13:23:48.302705364 [W:onnxruntime:Default, device_discovery.cc:325 DiscoverDevicesForPlatform] GPU device discovery failed: device_discovery.cc:92 ReadFileContents Failed to open file: "/sys/class/drm/card0/device/vendor"


## Step 1 - Configuration

We choose one CHiRPE checkpoint (`bert`) and define output files for this tutorial run.

We keep all outputs inside `outputs/notebook_direct_fused/bert/` so nothing in the main pipeline is overwritten.

In [2]:
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src").exists():
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / "src").exists():
    raise RuntimeError("Run this notebook from repo root or notebooks/.")

BACKBONE = "bert"
MAX_LENGTH = 128
OPSET = 18

CHECKPOINT_DIR = REPO_ROOT / "outputs" / "string_onnx_checkpoints" / BACKBONE / "best_model"
OUT_DIR = REPO_ROOT / "outputs" / "notebook_direct_fused" / BACKBONE
OUT_DIR.mkdir(parents=True, exist_ok=True)

CLASSIFIER_ONNX = OUT_DIR / "classifier.onnx"
TOKENIZER_ONNX = OUT_DIR / "tokenizer.onnx"
FUSED_ONNX = OUT_DIR / "fused.onnx"
METADATA_JSON = OUT_DIR / "metadata.json"

if not CHECKPOINT_DIR.exists():
    raise FileNotFoundError("Checkpoint directory not found.")

def rel(path: Path) -> str:
    return str(path.relative_to(REPO_ROOT))

print(f"Backbone: {BACKBONE}")
print(f"Checkpoint: {rel(CHECKPOINT_DIR)}")
print(f"Output dir: {rel(OUT_DIR)}")

Backbone: bert
Checkpoint: outputs/string_onnx_checkpoints/bert/best_model
Output dir: outputs/notebook_direct_fused/bert


## Step 2 - Load tokenizer and classifier

We load the exact artifacts CHiRPE already trained/saved.

- tokenizer: used for text to tokens
- sequence classifier: predicts class logits

In [3]:
tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT_DIR)
try:
    model = AutoModelForSequenceClassification.from_pretrained(
        CHECKPOINT_DIR,
        attn_implementation="eager",
    )
except (TypeError, ValueError):
    model = AutoModelForSequenceClassification.from_pretrained(CHECKPOINT_DIR)

model.eval()
print("Loaded tokenizer + classifier")
print(f"num_labels: {model.config.num_labels}")

Loaded tokenizer + classifier
num_labels: 2


## Step 3 - Export classifier to ONNX

Why a wrapper?

Hugging Face models return objects (with multiple fields).
For serving, we usually need only `logits`.

So we wrap the model and export a cleaner graph with just one output tensor.

In [4]:
dummy_batch = tokenizer(
    [
        "The participant reports suspiciousness and mild confusion.",
        "The participant feels stable and reports no unusual experiences.",
    ],
    padding=True,
    truncation=True,
    max_length=MAX_LENGTH,
    return_tensors="pt",
)

accepted = set(inspect.signature(model.forward).parameters)
input_names = [name for name in tokenizer.model_input_names if name in dummy_batch and name in accepted]
if "input_ids" not in input_names:
    raise RuntimeError("input_ids not found for export")

class LogitsOnly(torch.nn.Module):
    def __init__(self, m, names):
        super().__init__()
        self.m = m
        self.names = names

    def forward(self, *args):
        kwargs = {name: val for name, val in zip(self.names, args)}
        return self.m(**kwargs).logits

wrapper = LogitsOnly(model, input_names).eval()
args = tuple(dummy_batch[name] for name in input_names)

dynamic_axes = {name: {0: "batch_size", 1: "sequence_length"} for name in input_names}
dynamic_axes["logits"] = {0: "batch_size"}

with torch.no_grad():
    torch.onnx.export(
        wrapper,
        args,
        str(CLASSIFIER_ONNX),
        input_names=input_names,
        output_names=["logits"],
        dynamic_axes=dynamic_axes,
        dynamo=False,
        opset_version=OPSET,
        do_constant_folding=True,
    )

onnx.checker.check_model(onnx.load(str(CLASSIFIER_ONNX)))
print("Classifier ONNX exported")
print("Inputs:", input_names)
print("File:", rel(CLASSIFIER_ONNX))

/tmp/ipykernel_18939/1850518100.py:34: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


Classifier ONNX exported
Inputs: ['input_ids', 'token_type_ids', 'attention_mask']
File: outputs/notebook_direct_fused/bert/classifier.onnx


In [5]:
classifier_session = ort.InferenceSession(str(CLASSIFIER_ONNX), providers=["CPUExecutionProvider"])
print("Classifier ONNX inputs:")
for item in classifier_session.get_inputs():
    print({"name": item.name, "type": item.type, "shape": item.shape})
print("Classifier ONNX outputs:")
for item in classifier_session.get_outputs():
    print({"name": item.name, "type": item.type, "shape": item.shape})

Classifier ONNX inputs:


{'name': 'input_ids', 'type': 'tensor(int64)', 'shape': ['batch_size', 'sequence_length']}
{'name': 'token_type_ids', 'type': 'tensor(int64)', 'shape': ['batch_size', 'sequence_length']}
{'name': 'attention_mask', 'type': 'tensor(int64)', 'shape': ['batch_size', 'sequence_length']}
Classifier ONNX outputs:
{'name': 'logits', 'type': 'tensor(float)', 'shape': ['batch_size', 2]}


## Step 4 - Export tokenizer to ONNX

`gen_processing_models(...)` from ORT Extensions creates a tokenizer graph.

Important: this graph uses custom ops, so later we must register the ORT Extensions shared library when creating an ONNX Runtime session.

In [6]:
tokenizer_model, _ = gen_processing_models(tokenizer, pre_kwargs={})
tokenizer_model.ir_version = onnx.load(str(CLASSIFIER_ONNX)).ir_version
onnx.save(tokenizer_model, str(TOKENIZER_ONNX))
onnx.checker.check_model(onnx.load(str(TOKENIZER_ONNX)))
print("Tokenizer ONNX exported")
print("File:", rel(TOKENIZER_ONNX))

Tokenizer ONNX exported
File: outputs/notebook_direct_fused/bert/tokenizer.onnx


In [7]:
ortx_path = Path(get_library_path())
so = ort.SessionOptions()
so.register_custom_ops_library(str(ortx_path))
tokenizer_session = ort.InferenceSession(str(TOKENIZER_ONNX), so, providers=["CPUExecutionProvider"])

print("Tokenizer ONNX inputs:")
for item in tokenizer_session.get_inputs():
    print({"name": item.name, "type": item.type, "shape": item.shape})
print("Tokenizer ONNX outputs:")
for item in tokenizer_session.get_outputs():
    print({"name": item.name, "type": item.type, "shape": item.shape})

Tokenizer ONNX inputs:
{'name': 'text', 'type': 'tensor(string)', 'shape': [None]}
Tokenizer ONNX outputs:
{'name': 'input_ids', 'type': 'tensor(int64)', 'shape': [None]}
{'name': 'token_type_ids', 'type': 'tensor(int64)', 'shape': [None]}
{'name': 'attention_mask', 'type': 'tensor(int64)', 'shape': [None]}
{'name': 'offset_mapping', 'type': 'tensor(int64)', 'shape': [None, 2]}


## Step 5 - Merge tokenizer + classifier

Why an adapter graph?

- Tokenizer ONNX outputs are 1D token streams (`[seq_len]`)
- Classifier ONNX expects 2D tensors (`[batch_size, sequence_length]`)

So we add a small adapter that:

1. slices to `MAX_LENGTH`
2. unsqueezes to add batch dimension

Then we merge three graphs:

`tokenizer -> adapter -> classifier`

In [8]:
classifier_model = onnx.load(str(CLASSIFIER_ONNX))
tokenizer_model = onnx.load(str(TOKENIZER_ONNX))

def get_opset(model_proto):
    for imp in model_proto.opset_import:
        if imp.domain in ("", "ai.onnx"):
            return int(imp.version)
    raise RuntimeError("No ai.onnx opset found")

opset = get_opset(classifier_model)
tokenizer_model.ir_version = classifier_model.ir_version

classifier_input_names = [i.name for i in classifier_model.graph.input]
tokenizer_output_names = {o.name for o in tokenizer_model.graph.output}
missing = [name for name in classifier_input_names if name not in tokenizer_output_names]
if missing:
    raise RuntimeError(f"Tokenizer ONNX missing required outputs: {missing}")

adapter_inputs = []
adapter_outputs = []
adapter_nodes = []
io_tok_to_adapter = []
io_adapter_to_classifier = []

axes_zero = helper.make_tensor("axes_zero", TensorProto.INT64, [1], [0])
slice_starts = helper.make_tensor("slice_starts", TensorProto.INT64, [1], [0])
slice_ends = helper.make_tensor("slice_ends", TensorProto.INT64, [1], [MAX_LENGTH])
slice_axes = helper.make_tensor("slice_axes", TensorProto.INT64, [1], [0])
slice_steps = helper.make_tensor("slice_steps", TensorProto.INT64, [1], [1])

for name in classifier_input_names:
    tok_name = f"tok_{name}"
    tok_clip = f"tok_{name}_clip"
    cls_name = f"cls_{name}"

    adapter_inputs.append(helper.make_tensor_value_info(tok_name, TensorProto.INT64, ["seq_len"]))
    adapter_outputs.append(helper.make_tensor_value_info(cls_name, TensorProto.INT64, ["batch_size", "sequence_length"]))

    adapter_nodes.append(
        helper.make_node(
            "Slice",
            [tok_name, "slice_starts", "slice_ends", "slice_axes", "slice_steps"],
            [tok_clip],
            name=f"Slice_{name}",
        )
    )
    adapter_nodes.append(
        helper.make_node(
            "Unsqueeze",
            [tok_clip, "axes_zero"],
            [cls_name],
            name=f"Unsqueeze_{name}",
        )
    )

    io_tok_to_adapter.append((name, tok_name))
    io_adapter_to_classifier.append((cls_name, name))

adapter_graph = helper.make_graph(
    adapter_nodes,
    "TokenizerClassifierAdapter",
    adapter_inputs,
    adapter_outputs,
    [axes_zero, slice_starts, slice_ends, slice_axes, slice_steps],
)
adapter_model = helper.make_model(adapter_graph, opset_imports=[helper.make_opsetid("", opset)])
adapter_model.ir_version = classifier_model.ir_version

merged_tok_adapter = onnx.compose.merge_models(
    tokenizer_model,
    adapter_model,
    io_map=io_tok_to_adapter,
    inputs=["text"],
    outputs=[pair[0] for pair in io_adapter_to_classifier],
    name="tok_plus_adapter",
)
merged_tok_adapter.ir_version = classifier_model.ir_version

fused_model = onnx.compose.merge_models(
    merged_tok_adapter,
    classifier_model,
    io_map=io_adapter_to_classifier,
    inputs=["text"],
    outputs=[o.name for o in classifier_model.graph.output],
    name="fused_string_classifier",
)
fused_model.ir_version = classifier_model.ir_version

onnx.save(fused_model, str(FUSED_ONNX))
onnx.checker.check_model(onnx.load(str(FUSED_ONNX)))

metadata = {
    "backbone": BACKBONE,
    "max_length": MAX_LENGTH,
    "classifier_inputs": classifier_input_names,
    "fused_model": rel(FUSED_ONNX),
}
METADATA_JSON.write_text(json.dumps(metadata, indent=2))

print("Fused ONNX created")
print("File:", rel(FUSED_ONNX))

Fused ONNX created
File: outputs/notebook_direct_fused/bert/fused.onnx


## Step 6 - Run fused inference

Because the fused graph includes tokenizer custom ops, we must register ORT Extensions when creating the ONNX Runtime session.

Then inference is simple:

- send `np.array([text], dtype=object)`
- get logits
- apply softmax to get probabilities

In [9]:
ortx = Path(get_library_path())
if not ortx.exists():
    raise FileNotFoundError("ORT Extensions shared library not found")

so = ort.SessionOptions()
so.register_custom_ops_library(str(ortx))
session = ort.InferenceSession(str(FUSED_ONNX), so, providers=["CPUExecutionProvider"])

input_name = session.get_inputs()[0].name
output_name = session.get_outputs()[0].name

def softmax(logits: np.ndarray) -> np.ndarray:
    x = logits - logits.max(axis=-1, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=-1, keepdims=True)

label_map = {0: "Healthy", 1: "CHR-P"}
sample_text = "The participant reports suspiciousness and occasional confusion."

logits = session.run([output_name], {input_name: np.array([sample_text], dtype=object)})[0]
probs = softmax(logits)
pred = int(np.argmax(probs, axis=-1)[0])

result = {
    "text": sample_text,
    "prediction": pred,
    "label": label_map.get(pred, str(pred)),
    "confidence": float(probs[0, pred]),
    "logits": logits[0].tolist(),
    "probabilities": probs[0].tolist(),
}
print(json.dumps(result, indent=2))

{
  "text": "The participant reports suspiciousness and occasional confusion.",
  "prediction": 1,
  "label": "CHR-P",
  "confidence": 0.6713151335716248,
  "logits": [
    -0.8213791251182556,
    -0.10723988711833954
  ],
  "probabilities": [
    0.32868486642837524,
    0.6713151335716248
  ]
}


## Troubleshooting

- **Error about custom ops/tokenizer op not found**:
  - make sure `register_custom_ops_library(...)` is called before creating the ORT session.
- **Input type error**:
  - fused model expects string tensor input, use `np.array([text], dtype=object)`.
- **Shape mismatch errors**:
  - ensure adapter graph uses the same classifier input names and `MAX_LENGTH` as export assumptions.

## Mini glossary

- **ONNX opset**: version of ONNX operator definitions used by a model graph.
- **IR version**: ONNX model format version.
- **Custom op**: non-standard ONNX operator provided by an extension library.
- **Logits**: raw classifier scores before softmax.
- **Softmax**: converts logits into probabilities that sum to 1.